# Day 020 — Exercise 2: build_index

**Goal:** Implement `build_index(docs, collection_name, chunk_size, overlap)` that chunks each document, embeds each chunk with `nomic-embed-text`, and stores everything in a ChromaDB collection with `{source, chunk_index}` metadata. **One real Ollama embedding call per chunk** in the checks.

In [ ]:
import ollama
import chromadb

## Provided: chunk_text and embed_text

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    if step <= 0:
        step = 1
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks


def embed_text(text: str, model: str = "nomic-embed-text") -> list[float]:
    return ollama.embeddings(model=model, prompt=text)["embedding"]

## Your Implementation

In [ ]:
def build_index(
    docs: dict,
    collection_name: str = 'second_brain',
    chunk_size: int = 300,
    overlap: int = 50,
):
    """
    Chunk all docs, embed each chunk, store in a ChromaDB collection.
    Returns the collection object.
    """
    # TODO: chromadb.Client() — create client
    # TODO: delete collection if it exists (try/except), then create_collection
    # TODO: for each source, text: chunk → embed → accumulate lists
    # TODO: collection.add(ids, embeddings, documents, metadatas)
    # TODO: ID format: f"{source}__{i}"
    # TODO: metadata: {'source': source, 'chunk_index': i}
    # TODO: return collection
    pass

## Check Your Work

In [ ]:
TEST_DOCS = {
    "note_a.txt": "Python is a versatile programming language used for data science and AI.",
    "note_b.txt": "Machine learning is a branch of artificial intelligence that learns from data.",
}

def _run_checks():
    total = 5
    passed = 0
    col = None

    # Check 1: defined
    try:
        assert 'build_index' in globals()
        passed += 1; print('✅ Check 1: build_index defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: returns a collection (2 embed calls)
    try:
        col = build_index(TEST_DOCS, collection_name='test_build_020', chunk_size=20, overlap=5)
        assert col is not None, 'build_index returned None'
        assert hasattr(col, 'query'), 'returned object has no .query method'
        passed += 1; print('✅ Check 2: build_index returns a collection')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: collection has documents
    try:
        assert col is not None, 'collection is None (Check 2 failed)'
        count = col.count()
        assert count > 0, f'collection is empty (count={count})'
        passed += 1; print(f'✅ Check 3: collection has {count} chunk(s)')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: metadata has 'source' key
    try:
        assert col is not None, 'collection is None'
        peek = col.peek(1)
        metas = peek['metadatas']
        assert len(metas) > 0 and 'source' in metas[0], f"metadata missing 'source': {metas}"
        passed += 1; print("✅ Check 4: metadata contains 'source' key")
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: collection is queryable (1 embed call)
    try:
        assert col is not None, 'collection is None'
        emb = embed_text('python programming')
        res = col.query(query_embeddings=[emb], n_results=1)
        assert len(res['documents'][0]) == 1
        passed += 1; print('✅ Check 5: collection is queryable')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def build_index(
    docs: dict,
    collection_name: str = "second_brain",
    chunk_size: int = 300,
    overlap: int = 50,
):
    client = chromadb.Client()
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    for source, text in docs.items():
        chunks = chunk_text(text, chunk_size, overlap)
        ids, embeddings, documents, metadatas = [], [], [], []
        for i, chunk in enumerate(chunks):
            ids.append(f"{source}__{i}")
            embeddings.append(embed_text(chunk))
            documents.append(chunk)
            metadatas.append({"source": source, "chunk_index": i})
        if ids:
            collection.add(
                ids=ids, embeddings=embeddings,
                documents=documents, metadatas=metadatas,
            )
    return collection
```

</details>